# 🏁 Enduro-ALE — Deep Q-Network (DQN) Agent

**Môi trường:** `ALE/Enduro-v5` (Atari 2600 via ALE/gymnasium)

Notebook này huấn luyện một agent DQN cải tiến để chơi trò chơi **Enduro** trên Atari.

## 🔑 Các cải tiến so với DQN gốc (DeepMind 2015)
| Thành phần | Vanilla DQN | Notebook này |
|---|---|---|
| Kiến trúc | DQN | **Dueling DQN** |
| Cập nhật target | Hard copy | Hard copy (mỗi 10k steps) |
| Chọn hành động | ε-greedy | ε-greedy (linear decay) |
| Optimizer | RMSprop | **Adam** (lr=2.5e-4) |
| Reward shaping | clip sign | clip sign |
| Biểu đồ | ❌ | **✅ Real-time & cuối huấn luyện** |



## 📦 1. Import Thư Viện


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random
import ale_py
import gymnasium as gym
from collections import deque
import torch.optim as optim
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython import display as ipydisplay

matplotlib.rcParams['figure.dpi'] = 100
plt.style.use('dark_background')

gym.register_envs(ale_py)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## ⚙️ 2. Siêu Tham Số (Hyperparameters)

Các siêu tham số được tối ưu dựa trên paper **DeepMind DQN (Mnih et al., 2015)** và các cải tiến thực nghiệm:

- `GAMMA = 0.99` — discount factor dài hạn
- `LR = 0.00025` — learning rate chuẩn cho Atari
- `EPSILON_DECAY = 250000` — đủ lớn để agent khám phá trước khi khai thác
- `TARGET_UPDATE = 10000` — giữ target network ổn định
- `WARMUP_STEPS = 10000` — pre-fill buffer trước khi học


In [ ]:
# ── Môi trường ──────────────────────────────────────────────────────────
ENV_NAME       = 'ALE/Enduro-v5'
FRAME_SKIP     = 4          # số frame bỏ qua (action repeat)

# ── Replay Buffer ────────────────────────────────────────────────────────
BUFFER_SIZE    = 100_000    # tăng từ 50k -> 100k để agent nhớ tốt hơn
BATCH_SIZE     = 32

# ── Học ──────────────────────────────────────────────────────────────────
GAMMA          = 0.99
LR             = 2.5e-4

# ── Epsilon-greedy ───────────────────────────────────────────────────────
EPSILON_START  = 1.0
EPSILON_END    = 0.05       # giảm từ 0.1 -> 0.05 để khai thác nhiều hơn
EPSILON_DECAY  = 250_000    # linear decay

# ── Target Network ───────────────────────────────────────────────────────
TARGET_UPDATE  = 10_000     # cập nhật target mỗi 10k optimization steps

# ── Training ─────────────────────────────────────────────────────────────
MAX_EPISODES   = 300
WARMUP_STEPS   = 10_000     # giảm từ 50k -> 10k (giữ buffer tối thiểu)
TRAIN_FREQ     = 4          # optimize mỗi 4 environment steps
LOG_EVERY      = 10         # in log mỗi N episodes
SAVE_EVERY     = 50         # lưu checkpoint mỗi N episodes

print("Hyperparameters loaded successfully.")
print(f"  Buffer: {BUFFER_SIZE:,} | Batch: {BATCH_SIZE} | LR: {LR}")
print(f"  Epsilon: {EPSILON_START} -> {EPSILON_END} over {EPSILON_DECAY:,} steps")
print(f"  Target update: every {TARGET_UPDATE:,} opt steps")


## 🖼️ 3. Tiền Xử Lý Frame (Preprocessing)

Mỗi frame RGB (210×160×3) được chuyển thành grayscale 84×84 — chuẩn của paper DQN.
4 frame liên tiếp được stack lại tạo tensor `(4, 84, 84)` để agent nhận biết chuyển động.


In [ ]:
def preprocess_frame(frame: np.ndarray) -> np.ndarray:
    """Chuyển frame RGB -> grayscale 84x84 (uint8)."""
    img = Image.fromarray(frame)
    img = img.convert('L')                       # grayscale
    img = img.resize((84, 84), Image.Resampling.LANCZOS)
    return np.array(img, dtype=np.uint8)


def make_state(frames_deque: deque) -> np.ndarray:
    """Stack 4 frame thành tensor (4, 84, 84)."""
    return np.array(frames_deque, dtype=np.uint8)


## 🗄️ 4. Replay Buffer

Experience Replay lưu trữ các transition `(s, a, r, s', done)` và lấy mẫu ngẫu nhiên
để phá vỡ tương quan giữa các transition liên tiếp, ổn định quá trình huấn luyện.


In [ ]:
class ReplayBuffer:
    """Circular replay buffer lưu trữ experience transitions."""

    def __init__(self, capacity: int = BUFFER_SIZE):
        self.memory = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int = BATCH_SIZE):
        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        s      = torch.tensor(np.array(states),      dtype=torch.float32).to(device)
        a      = torch.tensor(actions,               dtype=torch.long).to(device)
        r      = torch.tensor(rewards,               dtype=torch.float32).to(device)
        s_next = torch.tensor(np.array(next_states), dtype=torch.float32).to(device)
        d      = torch.tensor(dones,                 dtype=torch.float32).to(device)

        return s, a, r, s_next, d

    def __len__(self):
        return len(self.memory)


## 🧠 5. Kiến Trúc Mạng — Dueling DQN

**Dueling DQN** (Wang et al., 2016) tách luồng đầu ra thành hai nhánh:

- **Value stream** V(s): đánh giá trạng thái bất kể hành động nào
- **Advantage stream** A(s,a): lợi thế của từng hành động so với trung bình

$$Q(s,a) = V(s) + \left(A(s,a) - \frac{1}{|\mathcal{A}|} \sum_{a'} A(s,a')\right)$$

Cải tiến này giúp mạng học **ổn định hơn** cho các trạng thái mà hành động không quan trọng lắm.


In [ ]:
class DuelingDQN(nn.Module):
    """
    Dueling DQN:
    - Convolutional backbone chia sẻ giống DQN gốc (84x84 grayscale, 4-frame stack)
    - Tách thành Value head và Advantage head
    """

    def __init__(self, num_actions: int):
        super().__init__()
        # ── Shared Convolutional Backbone ─────────────────────────────────
        self.backbone = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4),  # -> (32,20,20)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), # -> (64, 9, 9)
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), # -> (64, 7, 7)
            nn.ReLU(),
            nn.Flatten(),                                # -> 3136
        )

        # ── Value Head V(s) ───────────────────────────────────────────────
        self.value_stream = nn.Sequential(
            nn.Linear(3136, 512),
            nn.ReLU(),
            nn.Linear(512, 1)
        )

        # ── Advantage Head A(s,a) ─────────────────────────────────────────
        self.advantage_stream = nn.Sequential(
            nn.Linear(3136, 512),
            nn.ReLU(),
            nn.Linear(512, num_actions)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x / 255.0                     # normalize pixel [0,255] -> [0,1]
        features = self.backbone(x)

        value     = self.value_stream(features)       # (B, 1)
        advantage = self.advantage_stream(features)   # (B, A)

        # Q(s,a) = V(s) + (A(s,a) - mean_a A(s,a))
        q = value + advantage - advantage.mean(dim=1, keepdim=True)
        return q


# Alias cho backward compat
DQN = DuelingDQN


## 🤖 6. Agent — Chọn Hành Động & Huấn Luyện

Agent sử dụng **ε-greedy policy**:
- Với xác suất ε: chọn hành động ngẫu nhiên (exploration)
- Còn lại: chọn hành động có Q-value cao nhất (exploitation)

ε giảm tuyến tính từ 1.0 → 0.05 trong `EPSILON_DECAY` steps.


In [ ]:
class Agent:
    """DQN Agent với Dueling architecture, Adam optimizer và gradient clipping."""

    def __init__(self, num_actions: int):
        self.num_actions  = num_actions
        self.steps_done   = 0   # tổng số optimization steps (dùng để update target)
        self.env_steps    = 0   # tổng số environment steps (dùng để tính epsilon)

        # ── Networks ──────────────────────────────────────────────────────
        self.policy_net = DuelingDQN(num_actions).to(device)
        self.target_net = DuelingDQN(num_actions).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        # ── Optimizer: Adam thay cho RMSprop (ổn định hơn) ────────────────
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=LR)
        self.loss_fn   = nn.SmoothL1Loss()  # Huber loss (robust hơn MSE)

    # ── Epsilon hiện tại (giảm tuyến tính theo env_steps) ─────────────────
    @property
    def epsilon(self) -> float:
        progress = min(self.env_steps / EPSILON_DECAY, 1.0)
        return EPSILON_START - progress * (EPSILON_START - EPSILON_END)

    def select_action(self, state: np.ndarray) -> int:
        self.env_steps += 1
        if random.random() < self.epsilon:
            return random.randrange(self.num_actions)
        with torch.no_grad():
            t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
            return self.policy_net(t).argmax(1).item()

    def optimize_model(self, buffer: ReplayBuffer) -> float | None:
        if len(buffer) < BATCH_SIZE:
            return None

        s, a, r, s_next, d = buffer.sample()

        # ── Current Q values ──────────────────────────────────────────────
        current_q = self.policy_net(s).gather(1, a.unsqueeze(1)).squeeze(1)

        # ── Target Q values (Double DQN) ──────────────────────────────────
        # policy_net chọn action, target_net đánh giá Q-value
        with torch.no_grad():
            best_actions = self.policy_net(s_next).argmax(1)       # action từ policy
            next_q = self.target_net(s_next).gather(            # value từ target
                1, best_actions.unsqueeze(1)
            ).squeeze(1)
            target_q = r + GAMMA * next_q * (1.0 - d)

        loss = self.loss_fn(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 10.0)
        self.optimizer.step()

        self.steps_done += 1

        # Hard update target network
        if self.steps_done % TARGET_UPDATE == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
            print(f"  [Step {self.steps_done:,}] Target network updated.")

        return loss.item()


## 📊 7. Hàm Vẽ Biểu Đồ

Các hàm tiện ích để quan sát quá trình huấn luyện:
- **Reward curve** — phần thưởng mỗi episode & trung bình trượt (moving average)
- **Loss curve** — Huber loss trong quá trình optimization
- **Epsilon decay** — lịch trình exploration
- **Q-value evolution** — giá trị Q trung bình (dấu hiệu mạng đang học)


In [ ]:
COLORS = {
    'reward':  '#00c6ff',
    'ma':      '#ff6b6b',
    'loss':    '#ffd700',
    'epsilon': '#a8ff78',
    'qval':    '#f953c6',
}


def moving_average(data: list, window: int = 20) -> np.ndarray:
    """Trung bình trượt."""
    if len(data) < window:
        return np.array(data)
    return np.convolve(data, np.ones(window) / window, mode='valid')


def plot_training(rewards: list, losses: list,
                  epsilons: list, q_values: list,
                  inline: bool = True):
    """
    Vẽ 4 biểu đồ tổng hợp:
    - Episode Reward (với moving average)
    - Training Loss
    - Epsilon Decay
    - Mean Q-Value
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle('Enduro DQN — Training Dashboard', fontsize=15, fontweight='bold', color='white')
    fig.patch.set_facecolor('#0d0d0d')
    for ax in axes.flat:
        ax.set_facecolor('#1a1a2e')

    episodes = list(range(1, len(rewards) + 1))

    # ── 1. Episode Reward ─────────────────────────────────────────────────
    ax = axes[0, 0]
    ax.plot(episodes, rewards, alpha=0.35, color=COLORS['reward'], linewidth=0.8, label='Reward')
    if len(rewards) >= 20:
        ma = moving_average(rewards, 20)
        ax.plot(range(20, len(rewards) + 1), ma, color=COLORS['ma'], linewidth=2, label='MA-20')
    ax.set_title('Episode Reward', color='white')
    ax.set_xlabel('Episode', color='#aaa'); ax.set_ylabel('Total Reward', color='#aaa')
    ax.tick_params(colors='#aaa'); ax.legend()
    ax.grid(alpha=0.15)

    # ── 2. Loss ───────────────────────────────────────────────────────────
    ax = axes[0, 1]
    if losses:
        loss_ep = list(range(1, len(losses) + 1))
        ax.plot(loss_ep, losses, color=COLORS['loss'], linewidth=0.9, alpha=0.8, label='Huber Loss')
        if len(losses) >= 20:
            ml = moving_average(losses, 20)
            ax.plot(range(20, len(losses) + 1), ml, color='white', linewidth=1.5, label='MA-20')
    ax.set_title('Training Loss (per episode avg)', color='white')
    ax.set_xlabel('Episode', color='#aaa'); ax.set_ylabel('Loss', color='#aaa')
    ax.tick_params(colors='#aaa'); ax.legend()
    ax.grid(alpha=0.15)

    # ── 3. Epsilon Decay ──────────────────────────────────────────────────
    ax = axes[1, 0]
    if epsilons:
        ax.plot(episodes, epsilons, color=COLORS['epsilon'], linewidth=1.5)
        ax.fill_between(episodes, epsilons, alpha=0.15, color=COLORS['epsilon'])
        ax.axhline(EPSILON_END, color='white', linestyle='--', linewidth=1, alpha=0.5, label=f'ε_min={EPSILON_END}')
    ax.set_title('Epsilon Decay', color='white')
    ax.set_xlabel('Episode', color='#aaa'); ax.set_ylabel('Epsilon', color='#aaa')
    ax.tick_params(colors='#aaa'); ax.legend()
    ax.grid(alpha=0.15)

    # ── 4. Mean Q-Value ───────────────────────────────────────────────────
    ax = axes[1, 1]
    if q_values:
        ax.plot(episodes, q_values, color=COLORS['qval'], linewidth=1.2, label='Mean Q')
        if len(q_values) >= 20:
            mq = moving_average(q_values, 20)
            ax.plot(range(20, len(q_values) + 1), mq, color='white', linewidth=1.5, label='MA-20')
    ax.set_title('Mean Q-Value (episode avg)', color='white')
    ax.set_xlabel('Episode', color='#aaa'); ax.set_ylabel('Q-value', color='#aaa')
    ax.tick_params(colors='#aaa'); ax.legend()
    ax.grid(alpha=0.15)

    plt.tight_layout()
    if inline:
        ipydisplay.clear_output(wait=True)
    plt.savefig('training_dashboard.png', dpi=120, bbox_inches='tight',
                facecolor='#0d0d0d')
    plt.show()


def plot_final_reward(rewards: list, window: int = 20):
    """Biểu đồ reward cuối cùng với histogram phân phối."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    fig.patch.set_facecolor('#0d0d0d')
    for ax in [ax1, ax2]:
        ax.set_facecolor('#1a1a2e')

    episodes = list(range(1, len(rewards) + 1))

    # Line plot
    ax1.plot(episodes, rewards, color=COLORS['reward'], alpha=0.4, linewidth=0.8)
    if len(rewards) >= window:
        ma = moving_average(rewards, window)
        ax1.plot(range(window, len(rewards) + 1), ma, color=COLORS['ma'],
                 linewidth=2.5, label=f'MA-{window}')
        ax1.axhline(np.mean(rewards), color='yellow', linestyle='--',
                    linewidth=1.5, alpha=0.7, label=f'Mean={np.mean(rewards):.1f}')
    ax1.set_title('Reward Curve — Final', color='white', fontsize=13)
    ax1.set_xlabel('Episode', color='#aaa'); ax1.set_ylabel('Reward', color='#aaa')
    ax1.tick_params(colors='#aaa'); ax1.legend()
    ax1.grid(alpha=0.15)

    # Histogram
    ax2.hist(rewards, bins=30, color=COLORS['reward'], alpha=0.7, edgecolor='#333')
    ax2.axvline(np.mean(rewards), color='yellow', linestyle='--', linewidth=2,
                label=f'Mean={np.mean(rewards):.1f}')
    ax2.axvline(np.median(rewards), color=COLORS['ma'], linestyle='-', linewidth=2,
                label=f'Median={np.median(rewards):.1f}')
    ax2.set_title('Reward Distribution', color='white', fontsize=13)
    ax2.set_xlabel('Reward', color='#aaa'); ax2.set_ylabel('Count', color='#aaa')
    ax2.tick_params(colors='#aaa'); ax2.legend()
    ax2.grid(alpha=0.15)

    plt.tight_layout()
    plt.savefig('reward_distribution.png', dpi=120, bbox_inches='tight',
                facecolor='#0d0d0d')
    plt.show()


## 🔥 8. Warmup Phase — Pre-fill Replay Buffer

Trước khi huấn luyện, ta thu thập `WARMUP_STEPS` transitions bằng policy ngẫu nhiên
để buffer có đủ dữ liệu đa dạng cho lần lấy mẫu đầu tiên.


In [ ]:
env    = gym.make(ENV_NAME, frameskip=FRAME_SKIP)
agent  = Agent(env.action_space.n)
buffer = ReplayBuffer(BUFFER_SIZE)

print(f"Environment: {ENV_NAME}")
print(f"Action space: {env.action_space.n} actions")
print(f"Observation shape: {env.observation_space.shape}")
print(f"Network parameters: {sum(p.numel() for p in agent.policy_net.parameters()):,}")
print()
print(f"--- WARMUP PHASE ({WARMUP_STEPS:,} steps) ---")

raw_frame, _ = env.reset()
frames        = deque([preprocess_frame(raw_frame)] * 4, maxlen=4)
current_state = make_state(frames)

for i in range(WARMUP_STEPS):
    action = env.action_space.sample()
    next_frame, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

    frames.append(preprocess_frame(next_frame))
    next_state = make_state(frames)

    buffer.push(current_state, action, float(np.sign(reward)), next_state, done)
    current_state = next_state

    if done:
        raw_frame, _ = env.reset()
        frames        = deque([preprocess_frame(raw_frame)] * 4, maxlen=4)
        current_state = make_state(frames)

    if (i + 1) % (WARMUP_STEPS // 5) == 0:
        print(f"  > Loaded {i + 1:,}/{WARMUP_STEPS:,} frames | Buffer: {len(buffer):,}")

print(f"\nWarmup complete. Buffer size: {len(buffer):,}")


## 🚀 9. Training Loop

Vòng lặp huấn luyện chính:
1. Agent tương tác với môi trường, thu thập transitions
2. Mỗi `TRAIN_FREQ` bước: tối ưu hóa policy network
3. Mỗi `TARGET_UPDATE` optimization steps: cập nhật target network
4. Log và vẽ biểu đồ mỗi `LOG_EVERY` episodes
5. Lưu checkpoint mỗi `SAVE_EVERY` episodes

> **Note:** Biểu đồ được cập nhật **inline** trong quá trình huấn luyện để quan sát progress.


In [ ]:
# ── Lịch sử huấn luyện ───────────────────────────────────────────────────
history = {
    'rewards':  [],   # tổng reward mỗi episode
    'losses':   [],   # loss trung bình mỗi episode
    'epsilons': [],   # epsilon cuối mỗi episode
    'q_values': [],   # mean Q cuối mỗi episode
}

print(f"--- TRAINING PHASE ({MAX_EPISODES} episodes) ---")

for episode in range(1, MAX_EPISODES + 1):
    raw_frame, _ = env.reset()
    frames        = deque([preprocess_frame(raw_frame)] * 4, maxlen=4)
    current_state = make_state(frames)

    total_reward  = 0.0
    episode_losses = []
    episode_qvals  = []
    step_in_ep     = 0
    done           = False

    while not done:
        action = agent.select_action(current_state)

        next_frame, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        frames.append(preprocess_frame(next_frame))
        next_state = make_state(frames)

        buffer.push(current_state, action, float(np.sign(reward)), next_state, done)
        current_state = next_state
        total_reward += reward
        step_in_ep   += 1

        # ── Optimize mỗi TRAIN_FREQ env steps ────────────────────────────
        if step_in_ep % TRAIN_FREQ == 0:
            loss_val = agent.optimize_model(buffer)
            if loss_val is not None:
                episode_losses.append(loss_val)

        # ── Theo dõi Q-value ──────────────────────────────────────────────
        if step_in_ep % 100 == 0 and len(buffer) >= BATCH_SIZE:
            with torch.no_grad():
                t = torch.tensor(current_state, dtype=torch.float32).unsqueeze(0).to(device)
                q_max = agent.policy_net(t).max().item()
                episode_qvals.append(q_max)

    # ── Ghi log episode ───────────────────────────────────────────────────
    avg_loss  = float(np.mean(episode_losses)) if episode_losses else 0.0
    avg_q     = float(np.mean(episode_qvals))  if episode_qvals  else 0.0
    eps_now   = agent.epsilon

    history['rewards'].append(total_reward)
    history['losses'].append(avg_loss)
    history['epsilons'].append(eps_now)
    history['q_values'].append(avg_q)

    if episode % LOG_EVERY == 0:
        recent_mean = np.mean(history['rewards'][-LOG_EVERY:])
        print(f"[Ep {episode:>4}/{MAX_EPISODES}] "
              f"Reward: {total_reward:>7.1f} | "
              f"Avg({LOG_EVERY}): {recent_mean:>7.1f} | "
              f"Loss: {avg_loss:.4f} | "
              f"Q: {avg_q:.3f} | "
              f"Eps: {eps_now:.4f}")

        # ── Vẽ biểu đồ inline (cập nhật mỗi LOG_EVERY ep) ────────────────
        plot_training(history['rewards'], history['losses'],
                      history['epsilons'], history['q_values'], inline=True)

    # ── Lưu checkpoint ────────────────────────────────────────────────────
    if episode % SAVE_EVERY == 0:
        ckpt_path = f"model_enduro_ep{episode}.pth"
        torch.save({
            'episode':    episode,
            'model_state_dict': agent.policy_net.state_dict(),
            'optimizer_state_dict': agent.optimizer.state_dict(),
            'history':    history,
        }, ckpt_path)
        print(f"  --> Checkpoint saved: {ckpt_path}")

env.close()
print()
print("=" * 60)
print("TRAINING COMPLETE")
print(f"  Best reward:  {max(history['rewards']):.1f}")
print(f"  Mean reward:  {np.mean(history['rewards']):.1f}")
print(f"  Final epsilon: {history['epsilons'][-1]:.4f}")
print("=" * 60)


## 📈 10. Biểu Đồ Cuối — Phân Tích Kết Quả

Sau khi huấn luyện xong, vẽ các biểu đồ tổng hợp để phân tích:
- Reward theo từng episode và phân phối
- Dashboard đầy đủ 4 metrics


In [ ]:
# ── Biểu đồ tổng hợp cuối cùng ──────────────────────────────────────────
print("--- FINAL TRAINING ANALYSIS ---")
plot_final_reward(history['rewards'], window=20)
plot_training(history['rewards'], history['losses'],
              history['epsilons'], history['q_values'], inline=False)

# ── Thống kê tóm tắt ─────────────────────────────────────────────────────
rewards_arr = np.array(history['rewards'])
print("\n== Summary Statistics ==")
print(f"  Episodes         : {len(rewards_arr)}")
print(f"  Best Reward      : {rewards_arr.max():.1f}")
print(f"  Mean Reward      : {rewards_arr.mean():.1f} ± {rewards_arr.std():.1f}")
print(f"  Median Reward    : {np.median(rewards_arr):.1f}")
print(f"  Last-50 Mean     : {rewards_arr[-50:].mean():.1f}")
print(f"  Total Opt Steps  : {agent.steps_done:,}")
print(f"  Total Env Steps  : {agent.env_steps:,}")


## 🎮 11. Kiểm Tra Agent — Ghi Video

Dùng model đã huấn luyện để chạy một episode hoàn chỉnh và ghi lại video.


In [ ]:
import os
import cv2
from IPython.display import Video, display

# ── Config ───────────────────────────────────────────────────────────────
MODEL_PATH  = "/kaggle/input/models/phmngcbch/650ep/pytorch/default/1/model_enduro_ep650.pth"
VIDEO_PATH  = "/kaggle/working/enduro_test.mp4"
FPS         = 30

# ── Load model ───────────────────────────────────────────────────────────
if os.path.exists(MODEL_PATH):
    ckpt = torch.load(MODEL_PATH, map_location=device)
    # Support both old (state_dict only) and new (dict with metadata) format
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        state_dict = ckpt['model_state_dict']
    else:
        state_dict = ckpt

    eval_env = gym.make(ENV_NAME, frameskip=FRAME_SKIP, render_mode='rgb_array')
    eval_agent = Agent(eval_env.action_space.n)
    eval_agent.policy_net.load_state_dict(state_dict, strict=False)
    eval_agent.policy_net.eval()
    print(f"Model loaded from: {MODEL_PATH}")
else:
    # Fallback: dùng agent vừa huấn luyện
    eval_env   = gym.make(ENV_NAME, frameskip=FRAME_SKIP, render_mode='rgb_array')
    eval_agent = agent
    eval_agent.policy_net.eval()
    print("Model path not found. Using freshly trained agent.")

# ── Record episode ────────────────────────────────────────────────────────
raw_frame, _ = eval_env.reset()
frame_h, frame_w = raw_frame.shape[:2]
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(VIDEO_PATH, fourcc, FPS, (frame_w, frame_h))

frames_q      = deque([preprocess_frame(raw_frame)] * 4, maxlen=4)
current_state = make_state(frames_q)
total_reward  = 0.0
done          = False

while not done:
    rgb = eval_env.render()
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    writer.write(bgr)

    with torch.no_grad():
        t = torch.tensor(current_state, dtype=torch.float32).unsqueeze(0).to(device)
        action = eval_agent.policy_net(t).argmax(1).item()

    next_frame, reward, terminated, truncated, _ = eval_env.step(action)
    done = terminated or truncated
    frames_q.append(preprocess_frame(next_frame))
    current_state = make_state(frames_q)
    total_reward += reward

writer.release()
eval_env.close()

print(f"Episode reward: {total_reward:.1f}")
print(f"Video saved to: {VIDEO_PATH}")
display(Video(VIDEO_PATH, embed=True))
